# Chapter 3 - Lab 6: LlamaIndex agent

**Goal**: build the agent with **LlamaIndex**'s `FunctionAgent`. LlamaIndex is data-centric: it shines when agents must search documents (Chapter 10).

This is the same reference task used across every framework lab in Chapter 3:
**compare the Zenith X1 smartphone (`ZX1`) and the FrostLine refrigerator (`FL-340`) on contact rate**
(support tickets per 100 delivered orders) and write a short memo for the Head of Customer Care.
Comparing all frameworks on the *same* task makes differences in API style, abstractions and ergonomics easy to spot.

*Finance edition*: compare Apple and JPMorgan on P/E. Contact rate plays the role of P/E: a ratio
that lets a high-volume and a low-volume product be compared fairly (ZX1 = 15.0, FL-340 = 36.0).

**Tech stack**:
* **LlamaIndex** (`llama-index`, `llama-index-llms-openai`).
* **OpenAI** `gpt-4o-mini`.

## 1. Install packages

In [ ]:
%pip install -q llama-index llama-index-llms-openai pydantic python-dotenv

## 2. Set up the API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared task setup

The task, tools, dataset and prompts live in `common.py` (next to this notebook in the repository).

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%203/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import (
    get_support_data,
    compute_contact_rate,
    support_data,
    system_message,
    input_message,
)

print("Tools loaded. Reference task:")
print(" ", input_message)

## 4. Agent

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

def support_data_tool(product_line: str) -> str:
    """Get the quarter's support tickets and delivered orders for a product line (e.g. ZX1, FL-340)."""
    d = get_support_data(product_line)
    return f"{product_line}: tickets={d.tickets} orders={d.orders}"

def contact_rate_tool(tickets: int, orders: int) -> float:
    """Compute the contact rate: support tickets per 100 orders."""
    return compute_contact_rate(tickets, orders)

agent = FunctionAgent(
    name="CXAnalyst",
    description="Computes and compares contact rates",
    system_prompt=system_message,
    tools=[support_data_tool, contact_rate_tool],
    llm=OpenAI(model="gpt-4o-mini"),
)

## 5. Run, streaming the tool calls

In [ ]:
from llama_index.core.agent.workflow import ToolCallResult

handler = agent.run(user_msg=input_message)
async for event in handler.stream_events():
    if isinstance(event, ToolCallResult):
        print(f"[{event.tool_name}] {event.tool_kwargs} -> {event.tool_output}")
response = await handler
print("\n[Memo]\n" + str(response))

## 6. Results
**What to notice about LlamaIndex**: workflow events give fine-grained visibility; the real strength appears with retrieval over policies and manuals (Chapter 10).